In [2]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

from lib.udaplay_vector_store import UdaPlayVectorStore

In [4]:
pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 173.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 39.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 32.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 37.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 36.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 48.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 42.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 60.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

### 1. vector store

In [5]:
vector_store = UdaPlayVectorStore(
    persist_dir="./chroma_db/udaplay_games",
    collection_name="udaplay_games",
    reset_collection=True,
)

print(vector_store.collection)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Collection(name=udaplay_games)


### 2. Load games

In [6]:
games_file = Path("data/games.json")
count = vector_store.add_games_from_file(games_file)
print(f"Added {count} game records to ChromaDB")

Added 6 game records to ChromaDB


### 3. Searching

In [7]:
test_queries = [
    "Who developed FIFA 21?",
    "When was God of War Ragnarok released?",
    "What platform was Pokémon Red launched on?",
    "Tell me about Rockstar games",
]

for query in test_queries:
    print("\n" + "=" * 90)
    print("QUERY:", query)
    results = vector_store.search_games(query, top_k=3)
    for i, result in enumerate(results, 1):
        meta = result["metadata"]
        print(f"\nResult {i}")
        print("Title:", meta.get("title"))
        print("Similarity:", result.get("similarity"))
        print("Source:", meta.get("source"))
        print(result["document"][:500])


QUERY: Who developed FIFA 21?

Result 1
Title: FIFA 21
Similarity: 0.7303
Source: data/games.json
Title: FIFA 21
Developer: EA Vancouver and EA Romania
Publisher: Electronic Arts
Release Date: October 9, 2020
Platforms: Microsoft Windows, PlayStation 4, Xbox One, Nintendo Switch
Genre: Sports
Description: FIFA 21 is a football simulation video game in the FIFA series.

Result 2
Title: Grand Theft Auto V
Similarity: 0.2446
Source: data/games.json
Title: Grand Theft Auto V
Developer: Rockstar North
Publisher: Rockstar Games
Release Date: September 17, 2013
Platforms: PlayStation 3, Xbox 360, PlayStation 4, Xbox One, Windows, PlayStation 5, Xbox Series X/S
Genre: Action-adventure
Description: Grand Theft Auto V is an open-world action-adventure game developed by Rockstar North.

Result 3
Title: Minecraft
Similarity: 0.1884
Source: data/games.json
Title: Minecraft
Developer: Mojang Studios
Publisher: Mojang Studios
Release Date: November 18, 2011
Platforms: Windows, macOS, Linux, Xbox, Pl

### 4. Single query

In [8]:
query = "Who developed FIFA 21?"
results = vector_store.search_games(query, top_k=1)
results[0]

{'id': 'game_97c9a9a6e97c3d63',
 'document': 'Title: FIFA 21\nDeveloper: EA Vancouver and EA Romania\nPublisher: Electronic Arts\nRelease Date: October 9, 2020\nPlatforms: Microsoft Windows, PlayStation 4, Xbox One, Nintendo Switch\nGenre: Sports\nDescription: FIFA 21 is a football simulation video game in the FIFA series.',
 'metadata': {'developer': 'EA Vancouver and EA Romania',
  'record_type': 'game',
  'title': 'FIFA 21',
  'source': 'data/games.json',
  'platforms': 'Microsoft Windows, PlayStation 4, Xbox One, Nintendo Switch',
  'release_date': 'October 9, 2020',
  'publisher': 'Electronic Arts',
  'genre': 'Sports'},
 'distance': 0.2696680426597595,
 'similarity': 0.7303}